In [1]:
import os
import qsprpred

from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP
import pandas as pd

/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [2]:
os.makedirs("dataset_outputs/CK1/data", exist_ok=True)

# Create dataset
dataset = QSPRDataset.fromTableFile(
    filename="data/src/datasets/ck1/data/CK1.csv",
    store_dir="dataset_outputs/CK1/data",
    name="CK1Dataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
)

In [3]:
dataset.prepareDataset(
    split=RandomSplit(test_fraction=0.2, dataset=dataset),
    feature_calculators=[MorganFP(radius=2, nBits=1024)],
    recalculate_features=True,
)

In [4]:
from qsprpred.data.descriptors.sets import RDKitDescs

rdkit_descs = RDKitDescs()

dataset.addDescriptors([rdkit_descs])

dataset.descriptorSets

In [5]:

# Přidejte cestu k vašemu lokálnímu repozitáři
import sys
import os

# Přidání cesty k lokálnímu repozitáři na začátek sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Zkontrolujte, zda je cesta v sys.path
print(sys.path)

from importlib import reload

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected
# Znovu načtěte modul, abyste zajistili, že je správně importován
reload(sys.modules['qsprpred.extra.gpu.models.neural_network'])

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected

os.chdir('/home/ubuntu/Bakalarka/QSPRpred')
print(os.getcwd())


import sys
import importlib.util

# Přidání cesty k repozitáři do sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Specifikujte cestu k souboru, který chcete importovat
module_path = '/home/ubuntu/Bakalarka/QSPRpred/qsprpred/extra/gpu/models/neural_network.py'
module_name = 'qsprpred.extra.gpu.models.neural_network'

# Načtěte modul z konkrétní cesty
spec = importlib.util.spec_from_file_location(module_name, module_path)
neural_network = importlib.util.module_from_spec(spec)
spec.loader.exec_module(neural_network)

# Nyní můžete používat třídu STFullyConnected
STFullyConnected = neural_network.STFullyConnected

['/home/ubuntu/Bakalarka/QSPRpred', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python311.zip', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/lib-dynload', '', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages']
/home/ubuntu/Bakalarka/QSPRpred
lol


In [6]:
from sklearn.model_selection import ParameterGrid
from torch.nn import functional as F
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
import pandas as pd
def test_fun(dic,  X_train, y_train, X_test, y_test) -> pd.DataFrame:
    param_grid_t = ParameterGrid(dic)
    i = 0
    val_f1_t = []
    val_acc_t = []
    param_len_t = len(param_grid_t)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device used:", device)
    for param in param_grid_t:
        i += 1
        print(i, '/', param_len_t)
        model_sts_t = STFullyConnected(n_dim=X_train.shape[1],  # počet vstupních neuronů (počet deskriptorů)
        n_class=1,  # regresní úloha (1 výstup)
        gpus=[],
        device=device,
        is_reg=False, **param)
        model_sts_t.fit(X_train, y_train)
        res = model_sts_t.predict(X_test)
        res = res >0.5
        val_f1_t.append(f1_score(res, y_test))
        val_acc_t.append(accuracy_score(res, y_test))
        print(param)
        print(f1_score(res, y_test))
        print(accuracy_score(res, y_test))
    my_df = pd.DataFrame(param_grid_t)
    my_df["F1"] = val_f1_t
    my_df["Acc"] = val_acc_t
    return my_df

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X = dataset.X
X1, X2, y1, y2 = train_test_split(dataset.X, dataset.y, test_size=0.25, random_state=42)
X3 = dataset.X_ind
y3 = dataset.y_ind

imp_mean = SimpleImputer(missing_values=pd.NA, strategy='mean')
X1 = imp_mean.fit_transform(X1)
X2 = imp_mean.transform(X2)
X3 = imp_mean.transform(X3)

scaler = StandardScaler()
X1 = scaler.fit_transform(X1)
X2 = scaler.transform(X2)
X3 = scaler.transform(X3)





In [8]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=0.5, random_state=42)
X1, y1 = smote.fit_resample(X1, y1)



In [9]:
display(pd.DataFrame(X1))

,0,1,2,3,4,5,6,7,8,9,...,1224,1225,1226,1227,1228,1229,1230,1231,1232,1233
0,-0.079057,-0.328326,-0.064483,-0.207838,0.0,-0.112154,0.0,-0.079057,-0.252738,-0.102275,...,-0.304566,-0.228665,-0.045549,-0.064483,-0.179029,0.0,-0.344345,-0.091382,-0.288027,-0.898713
1,-0.079057,-0.328326,-0.064483,-0.207838,0.0,-0.112154,0.0,-0.079057,-0.252738,-0.102275,...,-0.304566,-0.228665,-0.045549,-0.064483,-0.179029,0.0,-0.344345,-0.091382,-0.288027,0.749777
2,-0.079057,-0.328326,-0.064483,-0.207838,0.0,-0.112154,0.0,-0.079057,-0.252738,-0.102275,...,-0.304566,-0.228665,-0.045549,-0.064483,-0.179029,0.0,-0.344345,-0.091382,-0.288027,0.283185
3,-0.079057,3.045750,-0.064483,-0.207838,0.0,-0.112154,0.0,-0.079057,-0.252738,-0.102275,...,-0.304566,-0.228665,-0.045549,-0.064483,-0.179029,0.0,-0.344345,-0.091382,-0.288027,-1.905518
4,-0.079057,-0.328326,-0.064483,-0.207838,0.0,8.916277,0.0,-0.079057,3.956662,-0.102275,...,-0.304566,-0.228665,-0.045549,-0.064483,-0.179029,0.0,-0.344345,-0.091382,-0.288027,-0.677801
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
584,1.070166,-0.328326,-0.064483,-0.207838,0.0,-0.112154,0.0,-0.079057,-0.252738,-0.102275,...,-0.304566,-0.228665,-0.045549,-0.064483,-0.179029,0.0,-0.344345,-0.091382,-0.288027,1.002568
585,-0.079057,-0.328326,-0.064483,-0.207838,0.0,-0.112154,0.0,-0.079057,-0.252738,-0.102275,...,-0.304566,-0.228665,-0.045549,-0.064483,-0.179029,0.0,-0.344345,-0.091382,-0.288027,1.083523
586,-0.079057,-0.328326,-0.064483,-0.207838,0.0,-0.112154,0.0,-0.079057,-0.252738,-0.102275,...,-0.304566,-0.228665,-0.045549,-0.064483,3.736487,0.0,-0.344345,-0.091382,-0.288027,0.983938
587,-0.079057,-0.328326,-0.064483,-0.207838,0.0,-0.112154,0.0,-0.079057,-0.252738,-0.102275,...,-0.304566,-0.228665,-0.045549,-0.064483,-0.179029,0.0,-0.344345,-0.091382,-0.288027,0.830276


In [10]:
print("xd")
test_par = {'weight_decay': 0.001, 'patience': 50, 
            'neuron_layers': [5000, 2500],
            'n_epochs': 200, 'dropout_frac': 0.6,
            'act_fun': F.selu}
model_sts_3 = STFullyConnected(n_dim=X1.shape[1],  # počet vstupních neuronů (počet deskriptorů)
    n_class=1,  # regresní úloha (1 výstup)
    gpus=[],
    device="cuda",
    batch_size=256,is_reg=False, seed=69, **test_par)
model_sts_3.fit(X1, y1)
res = model_sts_3.predict(X2)
res = res >0.5
print(f1_score(res, y2))
# <function selu at 0x7f925b90b380>	256	0.6	200	(5000, 2500)	<class 'torch.optim.adamw.AdamW'>	75	0.00001	0.00100	0.580645	0.839506

xd
0.4927536231884058


In [11]:
from torch import optim
import torch
my_dict_ult = {
    "act_fun": [F.selu],
    "dropout_frac": [0.1, 0.4, 0.5, 0.6],
    "patience": [75],
    "tol": [1e-5],
    "weight_decay": [1e-4, 1],
    "n_epochs": [200,300, 400, 1000],
    "neuron_layers": [[ 5000, 2500], [ 1024, 512, 256, 128, 64, 32, 16, 8, 4],  [2048, 1024, 512, 256, 128], [4096, 2048], [7500, 3750]]
    , "batch_size": [256]
    , "optimizer": [optim.AdamW] 
}
df_batch_ult = test_fun(my_dict_ult, X1, y1, X2, y2)

Device used: cuda
1 / 160
{'act_fun': <function selu at 0x7f362d382d40>, 'batch_size': 256, 'dropout_frac': 0.1, 'n_epochs': 200, 'neuron_layers': [5000, 2500], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
0.5151515151515151
0.8024691358024691
2 / 160
{'act_fun': <function selu at 0x7f362d382d40>, 'batch_size': 256, 'dropout_frac': 0.1, 'n_epochs': 200, 'neuron_layers': [5000, 2500], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 1}
0.5151515151515151
0.8024691358024691
3 / 160
{'act_fun': <function selu at 0x7f362d382d40>, 'batch_size': 256, 'dropout_frac': 0.1, 'n_epochs': 200, 'neuron_layers': [1024, 512, 256, 128, 64, 32, 16, 8, 4], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
0.3036649214659686
0.17901234567901234
4 / 160
{'act_fun': <function selu at 0x7f362d382d40>, 'batch_size': 256, 'dropout_frac': 0.1, 'n_epochs': 200

KeyboardInterrupt: 

In [ ]:
df_batch_ult["neuron_layers"] = df_batch_ult["neuron_layers"].apply(lambda x: tuple(x) if isinstance(x, list) else x)
display(df_batch_ult.groupby("neuron_layers")["F1"].max())


In [ ]:
df_batch_ult.sort_values(by="F1", ascending=False)

In [ ]:
# <function selu at 0x7f925b90b380>	256	0.6	200	(5000, 2500)	<class 'torch.optim.adamw.AdamW'>	75	0.00001	0.00100	0.580645	0.839506

In [12]:
from torch import optim
import torch
my_dict_ult_final = {
    "act_fun": [F.selu],
    "dropout_frac": [0.65, 0.7, 0.75],
    "patience": [50],
    "tol": [0],
    "weight_decay": [  1e-4,0],
    "n_epochs": [100, 150, 200, 250],
    "neuron_layers": [[5000, 2500, 1000, 250], [5000, 2500, 1000]],
    "batch_size": [256, 128],
    "optimizer": [optim.AdamW, optim.RMSprop],
    "lr": [1e-4, 1e-3, 1e-5]
}
df_batch_ult_final = test_fun(my_dict_ult_final, X1, y1, X2, y2)

Device used: cuda
1 / 576
{'act_fun': <function selu at 0x7f362d382d40>, 'batch_size': 256, 'dropout_frac': 0.65, 'lr': 0.0001, 'n_epochs': 100, 'neuron_layers': [5000, 2500, 1000, 250], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 50, 'tol': 0, 'weight_decay': 0.0001}
0.4827586206896552
0.8148148148148148
2 / 576
{'act_fun': <function selu at 0x7f362d382d40>, 'batch_size': 256, 'dropout_frac': 0.65, 'lr': 0.0001, 'n_epochs': 100, 'neuron_layers': [5000, 2500, 1000, 250], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 50, 'tol': 0, 'weight_decay': 0}
0.4827586206896552
0.8148148148148148
3 / 576
{'act_fun': <function selu at 0x7f362d382d40>, 'batch_size': 256, 'dropout_frac': 0.65, 'lr': 0.0001, 'n_epochs': 100, 'neuron_layers': [5000, 2500, 1000, 250], 'optimizer': <class 'torch.optim.rmsprop.RMSprop'>, 'patience': 50, 'tol': 0, 'weight_decay': 0.0001}
0.4375
0.7777777777777778
4 / 576
{'act_fun': <function selu at 0x7f362d382d40>, 'batch_size': 256, 'dropo

In [ ]:
df_batch_ult_final.sort_values(by="F1", ascending =False) 
# <function selu at 0x7f925b90b380>	256	0.4	300	[4096, 2048]	<class 'torch.optim.adamw.AdamW'>	75	0.00001	0.00100	0.622951	0.858025

In [15]:
df_batch_ult_final.sort_values(by="F1", ascending=False, inplace=True)
df_batch_ult_final.to_csv('qsprpred/extra/gpu/models/dataset_outputs/CK1/tabs/ck1batch1.csv')

In [16]:
df_batch_ult_final

,act_fun,batch_size,dropout_frac,lr,n_epochs,neuron_layers,optimizer,patience,tol,weight_decay,F1,Acc
538,<function selu at 0x7f362d382d40>,128,0.75,0.001,250,"[5000, 2500, 1000, 250]",<class 'torch.optim.rmsprop.RMSprop'>,50,0,0.0001,0.600000,0.851852
350,<function selu at 0x7f362d382d40>,128,0.65,0.001,250,"[5000, 2500, 1000]",<class 'torch.optim.rmsprop.RMSprop'>,50,0,0.0001,0.597403,0.808642
327,<function selu at 0x7f362d382d40>,128,0.65,0.001,100,"[5000, 2500, 1000]",<class 'torch.optim.rmsprop.RMSprop'>,50,0,0.0000,0.593750,0.839506
519,<function selu at 0x7f362d382d40>,128,0.75,0.001,100,"[5000, 2500, 1000]",<class 'torch.optim.rmsprop.RMSprop'>,50,0,0.0000,0.583333,0.814815
54,<function selu at 0x7f362d382d40>,256,0.65,0.001,200,"[5000, 2500, 1000]",<class 'torch.optim.rmsprop.RMSprop'>,50,0,0.0001,0.571429,0.814815
...,...,...,...,...,...,...,...,...,...,...,...,...
248,<function selu at 0x7f362d382d40>,256,0.75,0.001,250,"[5000, 2500, 1000, 250]",<class 'torch.optim.adamw.AdamW'>,50,0,0.0001,0.354430,0.685185
137,<function selu at 0x7f362d382d40>,256,0.70,0.001,150,"[5000, 2500, 1000, 250]",<class 'torch.optim.adamw.AdamW'>,50,0,0.0000,0.345679,0.672840
41,<function selu at 0x7f362d382d40>,256,0.65,0.001,150,"[5000, 2500, 1000, 250]",<class 'torch.optim.adamw.AdamW'>,50,0,0.0000,0.345679,0.672840
136,<function selu at 0x7f362d382d40>,256,0.70,0.001,150,"[5000, 2500, 1000, 250]",<class 'torch.optim.adamw.AdamW'>,50,0,0.0001,0.345679,0.672840


In [21]:
my_dict_ult_final_2 = {
    "act_fun": [F.selu],
    "dropout_frac": [ 0.7, 0.75, 0.8],
    "patience": [50],
    "tol": [0],
    "weight_decay": [  1e-4],
    "n_epochs": [ 200, 250, 300, 350],
    "neuron_layers": [[5000, 2500, 1000, 250], [5000, 2500, 1000, 250, 125], [5000, 2500, 1000, 250, 50]],
    "batch_size": [256, 128, 64],
    "optimizer": [optim.AdamW],
    "lr": [1e-4, 1e-3, 1e-2]
}
df_batch_ult_final_2 = test_fun(my_dict_ult_final_2, X1, y1, X2, y2)

Device used: cuda
1 / 324


KeyboardInterrupt: 

In [ ]:
df_batch_ult_final_2.sort_values(by="F1", ascending=False, inplace=True)
df_batch_ult_final_2.to_csv('qsprpred/extra/gpu/models/dataset_outputs/CK1/tabs/ck1batch2.csv')